# EpistemicTrap — Analysis Report

This notebook summarizes benchmark results from exported `*.run.json` logs and produces a model-by-task score table with uncertainty estimates.


In [ ]:
import glob
import json
import math
import os
import random
from collections import defaultdict

random.seed(20260328)

def find_run_logs():
    paths = set()
    for pat in ['*.run.json', '**/*.run.json']:
        for p in glob.glob(pat, recursive=True):
            if os.path.isfile(p):
                paths.add(p)
    return sorted(paths)

def extract_numeric_results(run_json: dict):
    out = []
    for c in run_json.get('conversations', []) or []:
        r = (c.get('result') or {})
        numeric = (r.get('numericResult') or {})
        val = numeric.get('value')
        if val is None:
            continue
        try:
            out.append(float(val))
        except Exception:
            continue
    return out

def bootstrap_mean_ci(xs, iters=1000, alpha=0.05):
    if not xs:
        return None
    n = len(xs)
    means = []
    for _ in range(iters):
        sample = [xs[random.randrange(n)] for _ in range(n)]
        means.append(sum(sample) / n)
    means.sort()
    lo = means[int((alpha / 2) * iters)]
    hi = means[int((1 - alpha / 2) * iters) - 1]
    return (sum(xs) / n, lo, hi)

paths = find_run_logs()
print('Found run logs:', len(paths))
paths[:10]


In [ ]:
rows = []
for p in paths:
    with open(p, 'r', encoding='utf-8') as f:
        run = json.load(f)
    vals = extract_numeric_results(run)
    if not vals:
        continue
    task = (run.get('task') or {}).get('name') or run.get('taskName') or 'unknown_task'
    model = (run.get('model') or {}).get('name') or run.get('modelName') or 'unknown_model'
    rows.append((task, model, vals))

print('Parsed runs:', len(rows))
rows[:3]


In [ ]:
by_task_model = defaultdict(list)
for task, model, vals in rows:
    by_task_model[(task, model)].extend(vals)

summary = []
for (task, model), vals in sorted(by_task_model.items()):
    ci = bootstrap_mean_ci(vals)
    if not ci:
        continue
    mean, lo, hi = ci
    summary.append((task, model, len(vals), mean, lo, hi))

summary[:10]


In [ ]:
def print_table(rows, headers):
    cols = list(zip(*([headers] + rows))) if rows else [headers]
    widths = [max(len(str(x)) for x in col) for col in cols]
    fmt = ' | '.join('{:' + str(w) + '}' for w in widths)
    print(fmt.format(*headers))
    print('-+-'.join('-' * w for w in widths))
    for r in rows:
        print(fmt.format(*r))

table = []
for task, model, n, mean, lo, hi in summary:
    table.append((task, model, str(n), f"{mean:.3f}", f"{lo:.3f}", f"{hi:.3f}"))

print_table(table, headers=('task','model','n','mean','ci_lo','ci_hi'))


## Gradient Checks

Use this section to sanity-check that each task produces non-degenerate distributions (not all 0.0 or all 1.0) across models and across items.


In [ ]:
by_task = defaultdict(list)
for (task, model), vals in by_task_model.items():
    by_task[task].extend(vals)

degenerate = []
for task, vals in sorted(by_task.items()):
    mn = min(vals)
    mx = max(vals)
    mean = sum(vals) / len(vals)
    degenerate.append((task, len(vals), f"{mn:.3f}", f"{mx:.3f}", f"{mean:.3f}"))

print_table(degenerate, headers=('task','n','min','max','mean'))
